# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset on adoption predictors of indigenous and modern knowledge in rangeland management using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and is available at:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.
Here we list all available record sets and for each record set, its available fields (and their `@id`s), as defined in the loaded Croissant schema.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets are defined in the metadata. Attempting to load records directly from distributions.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- {rs['@id']} : {rs.get('name', '')}")
        if 'field' in rs:
            if isinstance(rs['field'], list):
                for fld in rs['field']:
                    print(f"    - field @id: {fld['@id']}")
            elif isinstance(rs['field'], dict):
                print(f"    - field @id: {rs['field']['@id']}")

## 3. Data Extraction
Load data from each available record set into a `pandas.DataFrame` for analysis. If no record sets are present, load from any available distribution directly.

In [ ]:
# Attempt to extract all available record sets into DataFrames by @id
record_sets = dataset.record_sets
dataframes = {}

if not record_sets:
    # If no record set, load all distributions directly (for this dataset, fields may exist as distributions)
    print("No record sets available in the schema. Attempting to load records from available distributions...")
    
    # List all distributions in metadata
    distributions = dataset.metadata.to_json().get('distribution', [])
    for dist in distributions:
        if isinstance(dist, dict):
            dist_id = dist.get('@id', None)
        else:
            dist_id = dist
        try:
            records = list(dataset.records(distribution=dist_id))
            df = pd.DataFrame(records)
            dataframes[dist_id] = df
            print(f"Loaded distribution {dist_id}: columns => {df.columns.tolist()}")
        except Exception as e:
            print(f"Could not load records for distribution {dist_id}: {e}")
    # Pick first loaded dataframe for demonstration
    if dataframes:
        first_dist_id = list(dataframes.keys())[0]
        print(f"\nPreview records from distribution {first_dist_id}:")
        display(dataframes[first_dist_id].head())
else:
    print("Record sets provided in schema. Loading each by @id...")
    record_set_ids = [rs['@id'] for rs in record_sets]
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id}: columns => {df.columns.tolist()}")
    # Preview first record set
    if dataframes:
        first_rs = record_set_ids[0]
        print(f"\nPreview records from record set {first_rs}:")
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping. Update field names using the loaded DataFrame columns. All processing refers to fields (columns) by their `@id`.

In [ ]:
# For demonstration, select the first DataFrame and its numeric columns
import numpy as np

if dataframes:
    # Use the first loaded DataFrame (either by record set or distribution)
    first_table_id = list(dataframes.keys())[0]
    df = dataframes[first_table_id]
    print(f"Columns in {first_table_id}:")
    print(df.columns.tolist())
    
    # Try to find a numeric column by inspecting types or by common column names
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Try to find by common names (e.g., 'log_likelihood', 'coefficient', etc.)
        for col in df.columns:
            if any(s in col.lower() for s in ['log', 'coef', 'value', 'std', 'error']):
                # Attempt to coerce to numeric
                df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
        # Choose a threshold (e.g., mean)
        threshold = df[numeric_field_id].mean(skipna=True)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (mean):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean(skipna=True)
        ) / filtered_df[numeric_field_id].std(skipna=True)
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try grouping by another column, prioritizing categorical candidate
        group_candidates = [c for c in df.columns if c not in numeric_candidates]
        group_field = None
        for candidate in group_candidates:
            if df[candidate].nunique() > 1 and df[candidate].nunique() < len(df) // 2:
                group_field = candidate
                break
        if group_field is not None:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields detected in the loaded DataFrame.")
else:
    print("No DataFrame loaded; cannot perform EDA.")

## 5. Visualization
Visualize the distribution of a chosen numeric field and, if grouping field exists, compare groups. Uses matplotlib and seaborn for convenient plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization for the first numeric and possible group field
if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
    
    # If grouping field exists, plot boxplot
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field or DataFrame available to plot.")

## 6. Conclusion
In this notebook, we demonstrated how to load a Croissant-described dataset with the `mlcroissant` library, explored its data organization via `@id`s, extracted tabular data, performed simple exploratory data analysis, and visualized relationships between variables. Further domain-specific analysis can build upon these foundations by leveraging the semantic information encoded with Croissant.